In [ ]:
!pip -q install datasets

In [ ]:
import os, json, random, shutil, time
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from datasets import load_dataset
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# ---------------- settings ----------------
BASE      = "distilroberta-base"   # alternatives: "roberta-base" (more accurate, ~2x slower), "microsoft/MiniLM-L12-H384-uncased" (faster)
EPOCHS    = 3
BATCH     = 32
LR        = 2e-5
MAX_LEN   = 128
QQP_TRAIN = 40000
SEED      = 42
OUT       = "/kaggle/working/verifier"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU found - training will be very slow. Enable the GPU accelerator.")

## 1. Data (same sources as notebook 1; label 1 = same meaning)

In [ ]:
def to_df(ds, a, b, src):
    return pd.DataFrame({"s1": ds[a], "s2": ds[b], "label": ds["label"], "src": src})

parts = {"train": [], "val": [], "test": []}
try:
    d = load_dataset("google-research-datasets/paws", "labeled_final")
    for k, sp in (("train", "train"), ("val", "validation"), ("test", "test")):
        parts[k].append(to_df(d[sp], "sentence1", "sentence2", "PAWS"))
    print("PAWS ok")
except Exception as e:
    print("PAWS skipped:", e)
try:
    d = load_dataset("nyu-mll/glue", "mrpc")
    for k, sp in (("train", "train"), ("val", "validation"), ("test", "test")):
        parts[k].append(to_df(d[sp], "sentence1", "sentence2", "MRPC"))
    print("MRPC ok")
except Exception as e:
    print("MRPC skipped:", e)
try:
    d = load_dataset("nyu-mll/glue", "qqp")
    q = to_df(d["train"], "question1", "question2", "QQP").sample(QQP_TRAIN + 4000, random_state=SEED).reset_index(drop=True)
    parts["train"].append(q.iloc[:QQP_TRAIN]); parts["val"].append(q.iloc[QQP_TRAIN:QQP_TRAIN+2000]); parts["test"].append(q.iloc[QQP_TRAIN+2000:])
    print("QQP ok")
except Exception as e:
    print("QQP skipped:", e)

assert parts["train"], "No dataset could be loaded - check that Internet is switched on."
train = pd.concat(parts["train"]).dropna().sample(frac=1, random_state=SEED).reset_index(drop=True)
val   = pd.concat(parts["val"]).dropna().reset_index(drop=True)
test  = pd.concat(parts["test"]).dropna().reset_index(drop=True)
train = train[train.label.isin([0, 1])].reset_index(drop=True)
val   = val[val.label.isin([0, 1])].reset_index(drop=True)
test  = test[test.label.isin([0, 1])].reset_index(drop=True)
print(len(train), len(val), len(test), "| positive rate:", round(train.label.mean(), 3))
print(test.src.value_counts().to_dict())

## 2. Model, prediction helper

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=2).to(device)

def encode(a, b):
    enc = tok(a, b, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return {k: v.to(device) for k, v in enc.items()}

@torch.no_grad()
def predict(df, bs=128):
    model.eval()
    order = np.argsort((df.s1.str.len() + df.s2.str.len()).values)   # similar lengths together = less padding
    probs = np.zeros(len(df), dtype=np.float32)
    for s in range(0, len(df), bs):
        idx = order[s:s + bs]
        b = df.iloc[idx]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(**encode(b.s1.tolist(), b.s2.tolist())).logits
        probs[idx] = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()
    return probs

## 3. Train

In [ ]:
os.makedirs(OUT, exist_ok=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps_per_epoch = (len(train) + BATCH - 1) // BATCH
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps_per_epoch * EPOCHS), steps_per_epoch * EPOCHS)
use_amp = device == "cuda"
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_auc = -1
for epoch in range(EPOCHS):
    model.train()
    order = np.random.permutation(len(train)); t0 = time.time(); run = 0.0
    for step in range(steps_per_epoch):
        b = train.iloc[order[step * BATCH:(step + 1) * BATCH]]
        y = torch.tensor(b.label.values, dtype=torch.long, device=device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(**encode(b.s1.tolist(), b.s2.tolist())).logits
        loss = F.cross_entropy(logits.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
        if (step + 1) % 300 == 0:
            print(f"epoch {epoch+1} step {step+1}/{steps_per_epoch} loss {run/300:.4f}"); run = 0.0
    auc = roc_auc_score(val.label.values, predict(val))
    print(f"== epoch {epoch+1} done in {time.time()-t0:.0f}s | val AUC {auc:.4f}")
    if auc > best_auc:
        best_auc = auc; model.save_pretrained(OUT); tok.save_pretrained(OUT); print("   saved best model")

## 4. Results on the test set

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(OUT).to(device)
p_val, p_test = predict(val), predict(test)

grid = np.linspace(0.05, 0.95, 91)
thr = float(grid[int(np.argmax([f1_score(val.label.values, p_val >= t) for t in grid]))])
# a stricter threshold (precision >= 0.85 on validation) for fewer false accusations
strict = next((float(t) for t in grid if precision_score(val.label.values, p_val >= t, zero_division=0) >= 0.85), thr)

def metrics(name, y, p, t):
    return {"model": name, "threshold": round(t, 2), "accuracy": round(accuracy_score(y, p >= t), 4),
            "precision": round(precision_score(y, p >= t, zero_division=0), 4), "recall": round(recall_score(y, p >= t), 4),
            "f1": round(f1_score(y, p >= t), 4), "auc": round(roc_auc_score(y, p), 4)}

rows = [
    {"model": "MiniLM bi-encoder, fine-tuned (notebook 1)", "threshold": 0.69, "accuracy": 0.7425, "precision": 0.6929,
     "recall": 0.8013, "f1": 0.7432, "auc": 0.8113},
    metrics("Cross-encoder (this notebook) @ best-F1 threshold", test.label.values, p_test, thr),
    metrics("Cross-encoder (this notebook) @ strict threshold", test.label.values, p_test, strict),
]
table = pd.DataFrame(rows); print(table.to_string(index=False))

print("\nAccuracy per dataset at the best-F1 threshold:")
per = {s: round(accuracy_score(test.label[test.src == s], p_test[(test.src == s).values] >= thr), 4) for s in test.src.unique()}
print(per)
table

In [ ]:
# quick sanity check on plagiarism-style examples (P = probability the two sentences mean the same)
@torch.no_grad()
def p(a, b):
    model.eval()
    return float(torch.softmax(model(**encode([a], [b])).logits, dim=-1)[0, 1])
for a, b in [("The cat sat on the mat.", "A cat was sitting on the mat."),
             ("Flights from New York to Florida are cheap.", "Flights from Florida to New York are cheap."),
             ("Photosynthesis converts light energy into chemical energy.", "Plants turn sunlight into stored chemical energy."),
             ("Photosynthesis converts light energy into chemical energy.", "The stock market fell sharply on Monday.")]:
    print(round(p(a, b), 3), "|", a, "<->", b)

## 5. Export for the app

In [ ]:
json.dump({"threshold": thr, "strict_threshold": strict, "base_model": BASE, "per_dataset_accuracy": per},
          open(os.path.join(OUT, "threshold.json"), "w"), indent=2)
json.dump(rows, open("/kaggle/working/crossenc_metrics.json", "w"), indent=2)
shutil.make_archive("/kaggle/working/crossenc", "zip", OUT)
print("Created crossenc.zip (" + str(round(os.path.getsize('/kaggle/working/crossenc.zip')/1e6)) + " MB):", os.listdir(OUT))

## 6. Use it in the app
1. Output tab -> download `crossenc.zip` (and `crossenc_metrics.json` for your report).
2. Unzip **into** `plagiarism-checker/models/verifier/` so that `models/verifier/config.json` and `threshold.json` exist.
3. `pip install torch transformers` (already installed if you use `sentence-transformers`).
4. Restart `python app.py` - the terminal prints `[verifier] loaded cross-encoder ...`.

The app uses `threshold` from `threshold.json`. For fewer false accusations, copy the `strict_threshold` value into `threshold`.